# **Correlation Analysis**

This notebook analyzes the correlation of cell intensities over time using tracked masks and extracted intensity data.

### **Import Libraries**
We begin by importing necessary packages, including:
- `pickle` for loading analysis results
- `tifffile` for reading/writing TIFF stacks
- `napari` for image and mask manipulation
- `cct_utils`, which contains our custom tracking logic

In [ ]:
import os
import tifffile
import numpy as np
import pickle
import napari
import matplotlib.pyplot as plt
import random
cct_utils = __import__('0_cct_utils')

### **Define Paths**
Set up all relevant paths for raw images, masks, and pickle files.  
Specify the model folder, file name, and output directories for plots and LaTeX tables.

Before proceeding, ensure that the required masks and `.pkl` files are generated by running the following notebooks:
- `1_generate_masks.ipynb` for mask generation
- `2_tracking_refinement.ipynb` for tracking refinement.

In [ ]:
# Get the parent directory of the current working directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))  # Moves up one level

# Define model name and base file name separately
model_folder = "ModelAB1"
file_name = 'OUA_140525_cluster2_20min_10i_340ms'

# Construct paths dynamically
raw_path = os.path.normpath(os.path.join(parent_dir, "raw_data", model_folder))
corr_path = os.path.normpath(os.path.join(parent_dir, "correlation_masks", model_folder))
pkl_path = os.path.normpath(os.path.join(parent_dir, "pkl_data", model_folder))
export_path = os.path.normpath(os.path.join(parent_dir, "plots"))
latex_path = os.path.normpath(os.path.join(parent_dir, "plots", "latex"))

# Dynamically construct full paths using file_name
raw_file = os.path.join(raw_path, f'{file_name}.tif')
corr_file = os.path.join(corr_path, f'{file_name}.tif')
pkl_file = os.path.join(pkl_path, f"{file_name}.pkl")

### **Step 1: Load Tracking Data**

In this step, we:
- Load the `.pkl` file containing cell tracking and intensity data.
- Print a warning if the file is missing.

In [ ]:
#Load pkl file
if os.path.exists(pkl_file):
    with open(pkl_file, 'rb') as f:
        d = pickle.load(f)
    print(f"Loaded {file_name}.pkl")
else:
    print(f"File not found: {pkl_file}")

### **Step 2: Load Preprocessed Mask**

In this step:
- Load the raw fluorescence image (`X`) and the **preprocessed mask** (`Y`) from the previous notebook.

In [ ]:
# Load preprocessed mask and raw image
X = tifffile.imread(raw_file)
Y = tifffile.imread(corr_file)  # Use pre-filtered mask from 2_tracking_refinement.ipynb

print(f"Loaded raw image {X.shape} and preprocessed mask {Y.shape}")

viewer = napari.Viewer()
viewer.add_image(X, name="Raw")
viewer.add_labels(Y, name="Preprocessed Mask", opacity=0.35)

# Ensure you define tracked_cells from Y directly
tracked_cells = np.unique(Y)
tracked_cells = tracked_cells[tracked_cells != 0]  # Remove background (0)
print(f"Tracked cells: {len(tracked_cells)}")

### **Step 3: Define Analysis Parameters**

In this step, we:
- Set parameters for frame rate, smoothing window size, maximum lag for correlation, and minimum overlap for correlation calculation.
- Specify output file names for saving plots.

In [ ]:
# Frame rate of the time series (frames per second)
fps = 1

# Applies a rolling window to smooth the intensity signal and reduce noise
window_size = 25  # Smooths over 25-second intervals (number of frames)

# Maximum time lag for cross-correlation analysis (in frames)
max_lag = 60  # Correlate up to 60 seconds apart

# Minimum overlap between signals for valid cross-correlation (in frames)
min_overlap = 100  # Minimum 100 overlapping frames (adjustable)

plot_name =  f"{file_name}_{window_size}.pdf"               #Name of the .png file to save the plot as, change end depending on window size chosen
plot_file_path = os.path.join(export_path, plot_name)       #Full path to the plot file

### **Step 4: Preprocess Intensity Data**

In this step, we:
- Normalize the intensity traces for each cell.
- Apply smoothing to the normalized traces.
- Plot both the normalized and filtered intensities for a random cell.
- Save the plot for reference.

In [ ]:
# Normalize intensities per cell
intensities_normalized = {
    cell_id: (d[cell_id]['intensities'] / np.max(d[cell_id]['intensities']) if np.max(d[cell_id]['intensities']) > 0 else d[cell_id]['intensities'])
    for cell_id in tracked_cells if cell_id in d
}

# Smooth normalized intensities
filtered_intensities = cct_utils.apply_smoothing_to_normalized(intensities_normalized, window_size, smoothing_method='padding')

# Plot for a random cell
random_cell_id = random.choice(list(intensities_normalized.keys()))
plt.figure(figsize=(10, 6))
plt.plot(filtered_intensities[random_cell_id], label='Filtered', color='red')
plt.plot(intensities_normalized[random_cell_id], label='Raw', color='blue', alpha=0.6)
plt.title(f'Intensity Curves for Cell {random_cell_id}')
plt.xlabel('Time (frames)')
plt.ylabel('Normalized Intensity')
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(export_path, f"{file_name}_{window_size}_normalized_intensity_curves.pdf"), bbox_inches='tight', dpi=300)
plt.show()

### **Step 5: Cross-Correlation Analysis**

In this step, we:
- Generates all possible pairs of tracked cells for correlation analysis.
- Calculates the cross-correlation between their intensity traces over time.
- Stores the maximum correlation value for each pair and prints the results.

In [ ]:
all_possible_pairs = [(cell1, cell2) for i, cell1 in enumerate(tracked_cells) for cell2 in tracked_cells[i+1:]]
all_correlations_filtered, max_correlations_filtered = [], []

for cell1, cell2 in all_possible_pairs:
    if cell1 in filtered_intensities and cell2 in filtered_intensities:
        corr_data = cct_utils.calculate_cross_correlation(cell1, cell2, max_lag, filtered_intensities, min_overlap=min_overlap)
        all_correlations_filtered.append(((cell1, cell2), corr_data))
        max_correlations_filtered.append(((cell1, cell2), np.nanmax(corr_data['correlations'])))

### **Step 6: Filter Cell Pairs by Distance and Correlation**

In this step:
- Filters the cell pairs based on a `distance threshold` (e.g., 100 pixels) to consider only spatially close cells.
- Calculates the cross-correlation for these filtered pairs and stores both maximum and minimum correlation values.
- Creates `LaTeX` tables summarizing the top 10 and bottom 10 pairs based on correlation values.

In [ ]:
distance_threshold = 100  # Adjust as needed

# Filter cell pairs based on distance and presence in d
filtered_pairs = [
    (cell1, cell2)
    for cell1, cell2 in all_possible_pairs
    if cell1 in d and cell2 in d and cct_utils.filter_cell_pairs_by_distance(cell1, cell2, d, threshold=distance_threshold)
]

# Prepare correlation containers
max_correlations_filtered, min_correlations_filtered = [], []

for cell1, cell2 in filtered_pairs:
    corr_data = cct_utils.calculate_cross_correlation(cell1, cell2, max_lag, filtered_intensities, min_overlap=min_overlap)
    max_correlations_filtered.append(((cell1, cell2), np.nanmax(corr_data['correlations'])))
    min_correlations_filtered.append(((cell1, cell2), np.nanmin(corr_data['correlations'])))
    
# Generate LaTeX tables for top 10 and bottom 10 correlations
cct_utils.generate_latex_table(max_correlations_filtered[:10], "Top 10", f"{file_name}_Top10", file_name, latex_path, min_correlations_filtered)
cct_utils.generate_latex_table(max_correlations_filtered[-10:], "Bottom 10", f"{file_name}_Bottom10", file_name, latex_path, min_correlations_filtered)

### **Step 7: Compute the Null Distribution and Find Significant Threshold**

This step:
- Builds a null distribution of maximum correlation values by randomly shuffling intensity traces for the filtered cell pairs.
- Compares the observed correlations with this null distribution to compute a significance threshold at a chosen p-value (e.g., 0.05).

In [ ]:
null_distribution = cct_utils.build_null_distribution(filtered_pairs, filtered_intensities, max_lag, num_permutations=1000, min_overlap=min_overlap)
threshold, p_val = cct_utils.get_significant_correlation_threshold(sorted(max_correlations_filtered, key=lambda x: x[1]), null_distribution, p_value_threshold=0.05)

### **Step 8: Plot Cross-Correlation Curves and Intensity Curves**

In this step, we:
- Identifies cell pairs with correlation values above the significant threshold.
- Plots the cross-correlation curves and intensity curves (with lag shift) for these highly correlated cell pairs.

In [ ]:
significant_pairs = [(pair, corr) for pair, corr in max_correlations_filtered if corr > threshold]
for (cell1, cell2), _ in sorted(significant_pairs, key=lambda x: x[1], reverse=True):
    corr_data = next(cd for (c1, c2), cd in all_correlations_filtered if {c1, c2} == {cell1, cell2})
    cct_utils.plot_correlation_curve(corr_data, fps, cell1, cell2, max_lag)
    cct_utils.plot_intensity_curves(cell1, cell2, corr_data, fps, filtered_intensities)

### **Step 9: Calculate COM for Network and Non-Network Cells**

In this step, we:
- Calculate the center of mass (COM) for cells with significant correlations (`filtered_cell_coms_at_frame`) and those without (`non_network_cell_coms_at_frame`).
- The `calculate_and_save_cell_coms_at_frame` function computes valid COMs for each cell at `frame_to_use` by checking the frame and nearby frames (`max_frame_gap`).

In [ ]:
# Define frame to use and max_frame_gap
frame_to_use = 600
max_frame_gap = 250

# Extract labels from significant pairs
unique_labels_filtered = cct_utils.extract_unique_labels_from_pairs([pair for pair, corr in significant_pairs])

# Identify non-network cells
non_network_cells = list(set(tracked_cells) - set(unique_labels_filtered))
print(f"Non-network cells: {non_network_cells}")

# Compute COMs for network cells
filtered_cell_coms_at_frame = cct_utils.calculate_and_save_cell_coms_at_frame(Y, unique_labels_filtered, frame_to_use, max_frame_gap)

# Compute COMs for non-network cells
non_network_cell_coms_at_frame = cct_utils.calculate_and_save_cell_coms_at_frame(Y, non_network_cells, frame_to_use, max_frame_gap)

### **Step 10: Visualize Network and COMs in Napari**

This step:
- Visualizes the network connections between highly correlated cell pairs along with their COMs in Napari.
- Highlights both network and non-network cells using different colors for easy distinction.

In [ ]:
cct_utils.plot_network_and_cell_coms_in_napari(
    X, significant_pairs, filtered_cell_coms_at_frame, non_network_cell_coms_at_frame,
    frame_to_use, export_path=export_path, plot_file_path=plot_file_path
)